In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.telecom_silver.network_events")

gold_df = (
    silver_df
    .groupBy(
        "event_date",
        "region",
        "network_type"
    )
    .agg(
        count("*").alias("total_events"),
        avg("latency_ms").alias("avg_latency_ms"),
        avg("signal_strength").alias("avg_signal_strength"),
        sum("data_usage_mb").alias("total_data_usage_mb"),
        (avg("dropped_call") * 100).alias("dropped_call_pct")
    )
)
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.telecom_gold.network_kpi_daily")

In [0]:
%sql
SELECT *
FROM workspace.telecom_gold.network_kpi_daily
LIMIT 20;

event_date,region,network_type,total_events,avg_latency_ms,avg_signal_strength,total_data_usage_mb,dropped_call_pct
2026-06-03,Chicago,5G,395,257.5417721518987,-82.80506329113923,93862.76999999992,19.49367088607595
2026-06-08,Chicago,LTE,390,260.8871794871795,-82.3025641025641,89663.84999999996,26.923076923076923
2026-06-09,Dallas,4G,357,265.0644257703081,-85.82913165266106,83074.02,31.372549019607842
2026-06-09,Houston,LTE,338,252.13905325443787,-85.69822485207101,83280.61000000003,23.37278106508876
2026-06-08,Austin,4G,394,252.63959390862945,-86.26395939086295,98361.67999999996,27.918781725888326
2026-06-02,Houston,5G,18,281.72222222222223,-85.33333333333333,4654.5599999999995,33.33333333333333
2026-06-02,Chicago,LTE,17,206.11764705882354,-78.76470588235294,4154.710000000001,17.647058823529413
2026-06-02,Atlanta,5G,14,269.85714285714283,-87.21428571428571,3415.9,28.57142857142857
2026-06-06,New York,LTE,425,257.70588235294116,-85.27764705882353,105397.39000000006,25.41176470588235
2026-06-05,Austin,LTE,385,266.85974025974025,-85.48831168831168,94467.33000000002,23.116883116883116


In [0]:
%sql
SELECT *
FROM workspace.telecom_gold.network_kpi_daily
ORDER BY event_date, region, network_type
LIMIT 20;

event_date,region,network_type,total_events,avg_latency_ms,avg_signal_strength,total_data_usage_mb,dropped_call_pct
2026-06-02,Atlanta,4G,14,277.57142857142856,-86.14285714285714,2860.95,21.428571428571427
2026-06-02,Atlanta,5G,14,269.85714285714283,-87.21428571428571,3415.9,28.57142857142857
2026-06-02,Atlanta,LTE,19,282.10526315789474,-90.6842105263158,4442.89,26.31578947368421
2026-06-02,Austin,4G,16,303.5,-99.25,3998.3700000000003,6.25
2026-06-02,Austin,5G,19,244.78947368421052,-81.94736842105263,3160.3100000000004,31.57894736842105
2026-06-02,Austin,LTE,16,290.0625,-83.9375,5132.8,6.25
2026-06-02,Chicago,4G,17,287.4117647058824,-84.29411764705883,3735.4,17.647058823529413
2026-06-02,Chicago,5G,19,225.8421052631579,-92.10526315789474,3985.5,26.31578947368421
2026-06-02,Chicago,LTE,17,206.11764705882354,-78.76470588235294,4154.710000000001,17.647058823529413
2026-06-02,Dallas,4G,14,342.85714285714283,-86.71428571428571,2933.0600000000004,0.0


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

w = Window.partitionBy("event_date").orderBy(col("avg_latency_ms").desc())

worst_latency_df = (
    gold_df
    .withColumn("latency_rank", row_number().over(w))
    .filter(col("latency_rank") <= 5)
)

worst_latency_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.telecom_gold.top_latency_regions")

In [0]:
%sql
SELECT *
FROM workspace.telecom_gold.top_latency_regions
ORDER BY event_date, latency_rank;

event_date,region,network_type,total_events,avg_latency_ms,avg_signal_strength,total_data_usage_mb,dropped_call_pct,latency_rank
2026-06-02,Dallas,4G,14,342.85714285714283,-86.71428571428571,2933.0600000000004,0.0,1
2026-06-02,Austin,4G,16,303.5,-99.25,3998.3700000000003,6.25,2
2026-06-02,Austin,LTE,16,290.0625,-83.9375,5132.8,6.25,3
2026-06-02,Chicago,4G,17,287.4117647058824,-84.29411764705883,3735.4,17.647058823529413,4
2026-06-02,Dallas,LTE,15,283.1333333333333,-93.26666666666667,3466.14,6.666666666666667,5
2026-06-03,Austin,LTE,385,269.9324675324675,-85.52987012987013,96591.84999999985,20.259740259740262,1
2026-06-03,New York,4G,400,263.37,-83.675,97730.38,23.5,2
2026-06-03,Chicago,LTE,396,262.81060606060606,-83.87373737373737,99058.33999999994,28.535353535353536,3
2026-06-03,Atlanta,LTE,386,261.2979274611399,-86.32642487046633,97541.61999999994,25.647668393782386,4
2026-06-03,Atlanta,4G,367,257.8828337874659,-84.50681198910081,90540.34999999999,22.3433242506812,5


In [0]:
detail_df = spark.sql("""
DESCRIBE DETAIL workspace.telecom_gold.network_kpi_daily
""")

display(detail_df)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1a9c02aa-4619-4cbb-95ba-2b0677445268,workspace.telecom_gold.network_kpi_daily,null,,2026-06-12T04:05:10.734Z,2026-06-12T04:05:14.000Z,List(),List(),1,7085,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
OPTIMIZE workspace.telecom_gold.network_kpi_daily;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781292080169, 1781292081685, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"
